# Practical exercises on classical methods for image denoising

## Learning goals
- Build intuition for how noise type affects image structure and statistics.
- Compare classical denoisers: Gaussian/median filtering, Wiener filtering, and TV denoising.
- Tune hyperparameters and reason about bias-variance tradeoffs.
- Evaluate with PSNR/SSIM when clean references exist.
- Use visual and no-reference checks for user-uploaded images.

## Outline
- Setup and data loading
- Exercise 1 (noise models)
- Exercise 2 (Gaussian + median filters)
- Exercise 3 (Wiener filter)
- Exercise 4 (Total Variation denoising)
- Exercise 5 (own image)
- Optional extras: NLM and hyperparameter optimization based on a dataset

## Setup and helper functions

In [ ]:
import io
import time
import warnings

import numpy as np
import matplotlib.pyplot as plt

from scipy import ndimage
from scipy.signal import wiener

from skimage import data, img_as_float, color, transform, io as skio, util
from skimage.filters import gaussian
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from skimage.restoration import denoise_tv_chambolle, denoise_nl_means, estimate_sigma

try:
    from google.colab import files
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

print(f"NumPy version: {np.__version__}")
print(f"Running in Colab: {IN_COLAB}")


def load_demo_image(name="camera", target_size=None):
    demos = {
        "camera": data.camera(),
        "coins": data.coins(),
        "moon": data.moon(),
        "page": data.page(),
    }
    if name not in demos:
        raise ValueError(f"Unknown demo image: {name}. Choices: {list(demos.keys())}")

    img = img_as_float(demos[name])
    if img.ndim == 3:
        img = color.rgb2gray(img)

    if target_size is not None:
        img = transform.resize(img, target_size, anti_aliasing=True)

    return img.clip(min=0.0, max=1.0)


def upload_own_image(target_size=None):
    if not IN_COLAB:
        print("Upload helper is designed for Colab. Running fallback behavior (returns None).")
        return None

    uploaded = files.upload()
    if not uploaded:
        print("No file uploaded.")
        return None

    first_name = list(uploaded.keys())[0]
    raw = uploaded[first_name]
    image = skio.imread(io.BytesIO(raw))

    if image.ndim == 3:
        image = color.rgb2gray(image)

    image = img_as_float(image)
    if target_size is not None:
        image = transform.resize(image, target_size, anti_aliasing=True)
    return image.clip(min=0.0, max=1.0)


def add_noise(clean_image, noise_type="gaussian", amount=0.08, random_seed=7):
    rng = np.random.default_rng(random_seed)
    clean_image = clean_image.clip(min=0.0, max=1.0)

    if noise_type == "gaussian":
        noisy = clean_image + rng.normal(loc=0.0, scale=amount, size=clean_image.shape)
    elif noise_type == "poisson":
        # Scale the levels inversely with the amount and clip to 1e6 to avoid overflow
        levels = 1.0 / max(amount, 1e-6)
        noisy = rng.poisson(clean_image * levels) / levels
    elif noise_type in ("salt_pepper", "s&p", "salt-and-pepper"):
        noisy = util.random_noise(clean_image, mode="s&p", amount=amount, rng=rng)
    else:
        raise ValueError("noise_type must be one of: gaussian, poisson, salt_pepper")

    return noisy.clip(min=0.0, max=1.0)


def compute_metrics(reference, estimate):
    reference = reference.clip(min=0.0, max=1.0)
    estimate = estimate.clip(min=0.0, max=1.0)
    psnr = peak_signal_noise_ratio(reference, estimate, data_range=1.0)
    ssim = structural_similarity(reference, estimate, data_range=1.0)
    return {"PSNR": psnr, "SSIM": ssim}


def no_reference_summaries(image):
    image = image.clip(min=0.0, max=1.0)
    contrast = float(np.std(image))

    gx = np.diff(image, axis=1, prepend=image[:, :1])
    gy = np.diff(image, axis=0, prepend=image[:1, :])
    grad_mag = np.sqrt(gx**2 + gy**2)
    sharpness = float(np.mean(grad_mag))

    return {
        "contrast_std": contrast,
        "gradient_sharpness": sharpness,
        "mean_intensity": float(np.mean(image)),
    }


def show_comparison(images, titles, cmap="gray", figsize=(15, 4), zoom=64):
    n = len(images)
    _, axes = plt.subplots(2, n, figsize=figsize, constrained_layout=True)

    for i, (img, title) in enumerate(zip(images, titles)):
        img = img.clip(min=0.0, max=1.0)
        axes[0, i].imshow(img, cmap=cmap, vmin=0, vmax=1)
        axes[0, i].set_title(title)
        axes[0, i].axis("off")

        h, w = img.shape
        y0, y1 = h // 2 - zoom // 2, h // 2 + zoom // 2
        x0, x1 = w // 2 - zoom // 2, w // 2 + zoom // 2
        crop = img[max(0, y0):min(h, y1), max(0, x0):min(w, x1)]
        axes[1, i].imshow(crop, cmap=cmap, vmin=0, vmax=1)
        axes[1, i].set_title(f"{title} (center crop)")
        axes[1, i].axis("off")

    plt.show()


def run_denoisers(noisy, methods):
    outputs = {}
    for name, fn in methods.items():
        t0 = time.perf_counter()
        result = fn(noisy)
        t1 = time.perf_counter()
        outputs[name] = {
            "image": result.clip(min=0.0, max=1.0),
            "runtime_s": t1 - t0,
        }
    return outputs

## Exercise 1: Load a demo image and inspect noise effects

Pick a demo image and one noise type, then compare profiles and histograms.

TODOs:
- Choose `demo_name` from `camera`, `coins`, `moon`, `page`.
- Choose `noise_type` from `gaussian`, `poisson`, `salt_pepper`.
- Tune `noise_amount` and observe how the histogram and line profile change.

In [ ]:
# Exercise 1 code
# TODO: change these parameters.
clean_image = load_demo_image(name="camera")
# clean_image = upload_own_image()  # Uncomment to upload your own image in Colab
noise_type = "gaussian"
noise_amount = 0.1

noisy = add_noise(clean_image, noise_type=noise_type, amount=noise_amount, random_seed=7)

row = clean_image.shape[0] // 2
x = np.arange(clean_image.shape[1])

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
axes[0, 0].imshow(clean_image, cmap="gray", vmin=0, vmax=1)
axes[0, 0].set_title("Clean")
axes[0, 0].axis("off")

axes[0, 1].imshow(noisy, cmap="gray", vmin=0, vmax=1)
axes[0, 1].set_title(f"Noisy ({noise_type}, amount={noise_amount})")
axes[0, 1].axis("off")

axes[1, 0].plot(x, clean_image[row], label="clean", lw=2)
axes[1, 0].plot(x, noisy[row], label="noisy", alpha=0.8)
axes[1, 0].set_title("Center-row intensity profile")
axes[1, 0].set_xlabel("x")
axes[1, 0].set_ylabel("intensity")
axes[1, 0].legend()

axes[1, 1].hist(clean_image.ravel(), bins=64, alpha=0.6, label="clean")
axes[1, 1].hist(noisy.ravel(), bins=64, alpha=0.6, label="noisy")
axes[1, 1].set_title("Pixel histogram")
axes[1, 1].set_xlabel("intensity")
axes[1, 1].set_ylabel("count")
axes[1, 1].legend()

plt.show()

print("Noisy image no-reference summary:")
stats = no_reference_summaries(noisy)
print(f"contrast = {stats['contrast_std']:.4f} | sharpness = {stats['gradient_sharpness']:.4f} | mean = {stats['mean_intensity']:.4f}")

## Exercise 2: Gaussian and median filtering

Tune local-filter hyperparameters and compare smoothing vs edge preservation.

TODOs:
- Set `gaussian_sigma`.
- Set `median_size` (odd integer).
- Compare PSNR/SSIM and visual sharpness.

In [ ]:
# Exercise 2 code
# TODO: tune these values.
gaussian_sigma = 1.2
median_size = 3

methods_ex2 = {
    "Noisy": lambda x: x,
    "Gaussian": lambda x: gaussian(x, sigma=gaussian_sigma, preserve_range=True),
    "Median": lambda x: ndimage.median_filter(x, size=median_size),
}

results_ex2 = run_denoisers(noisy, methods_ex2)

images = [clean_image] + [results_ex2[k]["image"] for k in ["Noisy", "Gaussian", "Median"]]
titles = ["Clean", "Noisy", f"Gaussian (sigma={gaussian_sigma})", f"Median (size={median_size})"]
show_comparison(images, titles, figsize=(14, 6), zoom=96)

print("Metrics against clean reference:")
for name in ["Noisy", "Gaussian", "Median"]:
    m = compute_metrics(clean_image, results_ex2[name]["image"])
    print(f"{name:10s} | PSNR = {m['PSNR']:.2f} dB | SSIM = {m['SSIM']:.4f} | runtime = {results_ex2[name]['runtime_s']*1e3:.1f} ms")

## Exercise 3: Wiener filtering

Wiener filtering adapts local smoothing to estimated local variance.

Intuition: it looks at a small neighborhood around each pixel and asks whether the
variation in that window is more likely to come from real image structure or from noise.
In fairly uniform regions, most variation is probably noise, so the filter smooths more
aggressively. Near edges or textured areas, the local variance is higher, so it smooths
less to preserve meaningful detail.

You can think of it as a local, data-dependent compromise between averaging and keeping
structure: flat areas get cleaned up, while sharp transitions are protected more than
they would be with a plain blur.

TODOs:
- Set `wiener_mysize` (odd integer or tuple).
- Set `wiener_noise` (estimated noise power; try `None` first and then explicit values).

In [ ]:
# Exercise 3 code
# TODO: tune these values.
wiener_mysize = 5
wiener_noise = None  # try values like 0.005, 0.01, 0.02

methods_ex3 = {
    "Noisy": lambda x: x,
    "Wiener": lambda x: wiener(x, mysize=wiener_mysize, noise=wiener_noise),
}

results_ex3 = run_denoisers(noisy, methods_ex3)

images = [clean_image, results_ex3["Noisy"]["image"], results_ex3["Wiener"]["image"]]
titles = ["Clean", "Noisy", f"Wiener (mysize={wiener_mysize}, noise={wiener_noise})"]
show_comparison(images, titles, figsize=(11, 6), zoom=96)

print("Metrics against clean reference:")
for name in ["Noisy", "Wiener"]:
    m = compute_metrics(clean_image, results_ex3[name]["image"])
    print(f"{name:10s} | PSNR = {m['PSNR']:.2f} dB | SSIM = {m['SSIM']:.4f} | runtime = {results_ex3[name]['runtime_s']*1e3:.1f} ms")

## Exercise 4: Total variation denoising

TV denoising suppresses noise while encouraging piecewise-smooth regions.

TODOs:
- Set `tv_weight` and compare staircasing vs edge preservation.
- Optionally test `tv_max_num_iter` for convergence/runtime tradeoff.

In [ ]:
# Exercise 4 code
# TODO: tune these values.
tv_weight = 0.08
tv_max_num_iter = 200

methods_ex4 = {
    "Noisy": lambda x: x,
    "TV": lambda x: denoise_tv_chambolle(x, weight=tv_weight, max_num_iter=tv_max_num_iter),
}

results_ex4 = run_denoisers(noisy, methods_ex4)

images = [clean_image, results_ex4["Noisy"]["image"], results_ex4["TV"]["image"]]
titles = ["Clean", "Noisy", f"TV (weight={tv_weight}, iters={tv_max_num_iter})"]
show_comparison(images, titles, figsize=(11, 6), zoom=96)

print("Metrics against clean reference:")
for name in ["Noisy", "TV"]:
    m = compute_metrics(clean_image, results_ex4[name]["image"])
    print(f"{name:10s} | PSNR = {m['PSNR']:.2f} dB | SSIM = {m['SSIM']:.4f} | runtime = {results_ex4[name]['runtime_s']*1e3:.1f} ms")

## Exercise 5: Upload your own image and pick a best method

Use your own scientific image if available. For uploaded images, no clean reference
exists, so compare methods using visual inspection and no-reference summaries.

TODOs:
- Upload an image in Colab (RGB or grayscale).
- Tune one hyperparameter per method.
- Pick the method that best preserves the structure relevant to your scientific goal.

In [ ]:
# Exercise 5 code
own = upload_own_image()

if own is None:
    print("No uploaded image found. Using a fallback grayscale demo image.")
    own = load_demo_image("coins")

# Create a synthetic noisy variant so methods can be compared in this exercise setup.
own_noise_type = "gaussian"
own_noise_amount = 0.08
own_noisy = add_noise(own, noise_type=own_noise_type, amount=own_noise_amount, random_seed=13)

# TODO: adjust these parameters.
own_gaussian_sigma = 1.0
own_median_size = 3
own_wiener_mysize = 5
own_wiener_noise = None
own_tv_weight = 0.08

own_methods = {
    "Noisy": lambda x: x,
    "Gaussian": lambda x: gaussian(x, sigma=own_gaussian_sigma, preserve_range=True),
    "Median": lambda x: ndimage.median_filter(x, size=own_median_size),
    "Wiener": lambda x: wiener(x, mysize=own_wiener_mysize, noise=own_wiener_noise),
    "TV": lambda x: denoise_tv_chambolle(x, weight=own_tv_weight, max_num_iter=200),
}

own_results = run_denoisers(own_noisy, own_methods)

images = [own, own_results["Noisy"]["image"], own_results["Gaussian"]["image"], own_results["Median"]["image"], own_results["Wiener"]["image"], own_results["TV"]["image"]]
titles = ["Uploaded (clean unknown)", "Noisy", "Gaussian", "Median", "Wiener", "TV"]
show_comparison(images, titles, figsize=(18, 6), zoom=96)

print("No-reference summaries:")
for name in ["Noisy", "Gaussian", "Median", "Wiener", "TV"]:
    stats = no_reference_summaries(own_results[name]["image"])
    print(f"{name:10s} | contrast = {stats['contrast_std']:.4f} | sharpness = {stats['gradient_sharpness']:.4f} | mean = {stats['mean_intensity']:.4f}")

## Optional Exercise A: Non-local means (NLM)

NLM is a more advanced denoiser that often preserves texture better than purely local
filters.

Intuition: instead of averaging only the pixels right next to a target pixel, NLM looks
for **similar small patches** across a wider area of the image. If two patches have a
similar pattern, their center pixels are likely to represent the same underlying
structure, so NLM averages them together.

This helps because noise is mostly random, while real image structure tends to repeat.
By combining many pixels that belong to similar-looking patches, NLM can reduce noise
without blurring edges and fine texture as much as a Gaussian or median filter.

The main parameters control this tradeoff:

- `patch_size`: how large each compared patch is
- `patch_distance`: how far away NLM searches for similar patches
- `h`: how strict the similarity test is; larger values average more patches and give
  stronger smoothing, smaller values preserve more detail but may leave more noise

In [ ]:
# Optional Exercise A code: NLM
opt_clean = clean_image
opt_noisy = noisy

# NLM parameters (TODO: tune h and patch settings)
sigma_est = float(np.mean(estimate_sigma(opt_noisy, channel_axis=None)))
nlm_h = 0.8 * sigma_est
nlm_patch_size = 5
nlm_patch_distance = 6

nlm_out = denoise_nl_means(
    opt_noisy,
    h=nlm_h,
    patch_size=nlm_patch_size,
    patch_distance=nlm_patch_distance,
    fast_mode=True,
    channel_axis=None,
)

images = [opt_clean, opt_noisy, nlm_out]
titles = ["Clean", "Noisy", f"NLM (h={nlm_h:.4f})"]

show_comparison(images, titles, figsize=(16, 6), zoom=96)

for name, img in [("Noisy", opt_noisy), ("NLM", nlm_out)]:
    m = compute_metrics(opt_clean, img)
    print(f"{name:8s} | PSNR = {m['PSNR']:.2f} dB | SSIM = {m['SSIM']:.4f}")

## Optional Exercise B: Hyperparameter optimization based on a dataset

Use the publicly available `skimage.data.camera()` image as a reference dataset sample.

Goal:
- Optimize one method hyperparameter automatically (here: TV `weight`) by maximizing PSNR on synthetically noised data.

Extensions:
- Replace TV with Wiener (`mysize`, `noise`) or Gaussian (`sigma`).
- Run over multiple public images (`camera`, `coins`, `moon`) and average the metric.

In [ ]:
# Optional Exercise B code: optimize TV weight on a public dataset image
public_clean = load_demo_image("camera")
public_noisy = add_noise(public_clean, noise_type="gaussian", amount=0.10, random_seed=123)

# TODO: modify this candidate grid.
weight_grid = np.linspace(0.02, 0.24, 12)

records = []
for w in weight_grid:
    t0 = time.perf_counter()
    den = denoise_tv_chambolle(public_noisy, weight=float(w), max_num_iter=200)
    dt = time.perf_counter() - t0
    m = compute_metrics(public_clean, den)
    records.append((float(w), m["PSNR"], m["SSIM"], dt))

best = max(records, key=lambda x: x[1])
best_w = best[0]
best_img = denoise_tv_chambolle(public_noisy, weight=best_w, max_num_iter=200)

print("Top candidates by PSNR:")
for row in sorted(records, key=lambda x: x[1], reverse=True)[:5]:
    print(f"weight = {row[0]:.4f} | PSNR = {row[1]:.3f} | SSIM = {row[2]:.4f} | runtime = {row[3]*1e3:.1f} ms")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].plot([r[0] for r in records], [r[1] for r in records], marker="o")
axes[0].set_title("TV weight vs PSNR")
axes[0].set_xlabel("weight")
axes[0].set_ylabel("PSNR (dB)")

axes[1].plot([r[0] for r in records], [r[2] for r in records], marker="o", color="tab:orange")
axes[1].set_title("TV weight vs SSIM")
axes[1].set_xlabel("weight")
axes[1].set_ylabel("SSIM")
plt.show()

show_comparison(
    [public_clean, public_noisy, best_img],
    ["Public clean (camera)", "Public noisy", f"Best TV denoise (weight={best_w:.4f})"],
    figsize=(12, 6),
    zoom=96,
)

best_metrics = compute_metrics(public_clean, best_img)
print(f"Best TV weight = {best_w:.4f}")
print(f"Best PSNR = {best_metrics['PSNR']:.3f} dB")
print(f"Best SSIM = {best_metrics['SSIM']:.4f}")

## Wrap-up

Before finishing, answer:
1. Which method gave the best balance of denoising and structure preservation for your task?
2. Which hyperparameter had the strongest effect on output quality?
3. How did qualitative judgments differ from PSNR/SSIM rankings?

Optional next step:
- Repeat the workflow with different noise families and compare method rankings across conditions.